# 🌿 제주도 신재생에너지 발전량 예측
## Multi-Modal Fusion (ResNet18 CNN + LSTM) · KG-RAG Agent
> **2조 PRD v2.0** | Google Colab GPU 환경 | 위성 이미지 + 기상 수치 하이브리드 예측

---
### 목차
1. 환경 설정
2. 데이터 로딩 & 전처리 (기상 수치 + 위성 이미지)
3. Multi-Modal Dataset & DataLoader 구축
4. Multi-Modal Fusion 모델 아키텍처 (ResNet18 + LSTM)
5. 학습, 조기 종료 & 모델 평가
6. RAG 벡터 DB (FAISS) 구축
7. 지식 그래프 (NetworkX KG) 구축
8. 예측설명 에이전트 파이프라인


## 1. 환경 설정

In [ ]:
# Google Drive 마운트 (모델·데이터 영구 저장)
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/jeju_energy'
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f'{BASE_DIR}/models', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data/satellite', exist_ok=True) # 위성 이미지 보관 폴더
print("Drive 마운트 및 폴더 준비 완료:", BASE_DIR)

In [ ]:
# 필수 패키지 설치
!pip install faiss-cpu networkx matplotlib seaborn torchvision pillow -q
print("패키지 설치 완료")

In [ ]:
# 공통 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings, pickle, os
warnings.filterwarnings('ignore')

# 재현성 시드 고정
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

## 2. 데이터 로딩 & 전처리 (기상 수치 + 위성 이미지)

> **데이터 준비**:
> 1. `data/` 폴더에 8개 기상 변수 CSV 업로드
> 2. `data/satellite/` 폴더에 `천리안 2A호` 구름 위성 이미지를 각 시점 타임스탬프 이름(예: `20260603_1400.jpg`)으로 업로드해 주세요.
> 3. 만약 위성 이미지가 아직 없다면, 코드 내에서 **임의의 구름 형태 노이즈 이미지**를 자동 생성하여 학습 파이프라인의 에러가 나지 않도록 처리해 두었습니다.

In [ ]:
DATA_DIR = f'{BASE_DIR}/data'

WEATHER_FILES = {
    'wind_spd':   f'{DATA_DIR}/wind_velocity_jeju_230203_260603.csv',
    'wind_dir':   f'{DATA_DIR}/wind_direction_jeju_230203_260603.csv',
    'temp':       f'{DATA_DIR}/temperature_jeju_230203_260603.csv',
    'humidity':   f'{DATA_DIR}/humidity_jeju_230203_260603.csv',
    'pressure':   f'{DATA_DIR}/pressure_jeju_230203_260603.csv',
    'cloud':      f'{DATA_DIR}/cloud_cover_jeju_230203_260603.csv',
    'sunshine':   f'{DATA_DIR}/sunshine_jeju_230203_260603.csv',
    'solar_rad':  f'{DATA_DIR}/solar_radiation_jeju_230203_260603.csv',
 }

GENERATION_FILE = f'{DATA_DIR}/jeju_solar_wind_generation.csv'  # ★ 수정된 발전량 파일명

In [ ]:
# ── 기상 데이터 로딩, 병합 & 결측 보정 ───────────────────────────
def load_weather(files: dict) -> pd.DataFrame:
    dfs = []
    for col_name, path in files.items():
        try:
            df = pd.read_csv(path, encoding='utf-8-sig')
            df.columns = ['지점명', '일시', col_name]
            df['일시'] = pd.to_datetime(df['일시'])
            df = df[['일시', col_name]]
            dfs.append(df)
        except FileNotFoundError:
            print(f"⚠️ {col_name} CSV 파일을 찾을 수 없습니다. 더미 데이터를 생성합니다.")
            date_range = pd.date_range(start='2023-02-03 00:00:00', end='2026-06-03 23:00:00', freq='h')
            df = pd.DataFrame({'일시': date_range, col_name: np.random.uniform(10, 30, len(date_range))})
            dfs.append(df)
    
    merged = dfs[0]
    for df in dfs[1:]:
        merged = merged.merge(df, on='일시', how='outer')
    
    merged = merged.sort_values('일시').reset_index(drop=True)
    merged.rename(columns={'일시': 'timestamp'}, inplace=True)
    return merged

weather_df = load_weather(WEATHER_FILES)

# 풍향 -> sin/cos 분해
if 'wind_dir' in weather_df.columns:
    weather_df['wind_sin'] = np.sin(np.deg2rad(weather_df['wind_dir']))
    weather_df['wind_cos'] = np.cos(np.deg2rad(weather_df['wind_dir']))
    weather_df.drop(columns=['wind_dir'], inplace=True)
else:
    weather_df['wind_sin'] = np.random.uniform(-1, 1, len(weather_df))
    weather_df['wind_cos'] = np.random.uniform(-1, 1, len(weather_df))

# 야간 일조/일사 결측 처리
weather_df['sunshine']  = weather_df['sunshine'].fillna(0)
weather_df['solar_rad'] = weather_df['solar_rad'].fillna(0)
weather_df = weather_df.fillna(method='ffill').fillna(method='bfill')
print(f"기상 데이터로드 완료! 형상: {weather_df.shape}")

In [ ]:
# ── 발전량 실적 로딩 & 병합 (거래일, 시간 데이터 파싱 보정) ─────────────────
try:
    gen_df = pd.read_csv(GENERATION_FILE, encoding='utf-8-sig')
    
    # 2023.2.2 -> YYYY-MM-DD 변환
    gen_df['date'] = pd.to_datetime(gen_df['거래일'].str.replace('.', '-'))
    
    # 시간(1~24)을 00:00 ~ 23:00 형식 타임스탬프로 매칭하기 위해 (시간 - 1) 적용
    gen_df['hour'] = gen_df['시간'] - 1
    gen_df['timestamp'] = gen_df.apply(lambda r: r['date'] + pd.Timedelta(hours=int(r['hour'])), axis=1)
    
    # 컬럼명 통일: 태양광 -> solar_mw, 풍력 -> wind_mw
    gen_df.rename(columns={'태양광': 'solar_mw', '풍력': 'wind_mw'}, inplace=True)
    
    full_df = weather_df.merge(gen_df[['timestamp', 'solar_mw', 'wind_mw']], on='timestamp', how='inner')
    print(f"✅ 실측 발전량 병합 성공! 병합 데이터 형상: {full_df.shape}")
    
except FileNotFoundError:
    print("⚠️ 발전 실적 파일이 확인되지 않아 시뮬레이션용 가상 타겟 데이터를 만듭니다.")
    full_df = weather_df.copy()
    full_df['solar_mw'] = np.clip(full_df['solar_rad'] * 80 + np.random.normal(0, 3, len(full_df)), 0, 300)
    full_df['wind_mw'] = np.clip(full_df['wind_spd'] ** 2.2 * 1.8 + np.random.normal(0, 8, len(full_df)), 0, 450)

full_df = full_df.dropna().reset_index(drop=True)
print(f"최종 DataFrame 형상: {full_df.shape}")

In [ ]:
# ── 위성 이미지 압축 해제 및 로드 설정 (진행률 표시줄 적용) ──────────────────
import zipfile
import os
from tqdm.notebook import tqdm

SATELLITE_DIR = f'{BASE_DIR}/data/satellite'
zip_path = os.path.join(SATELLITE_DIR, 'jeju_dataset.zip')
extracted_dir = os.path.join(SATELLITE_DIR, 'jeju_dataset')

if os.path.exists(extracted_dir) and len(os.listdir(extracted_dir)) > 0:
    print(f"✅ 이미 압축 해제된 폴더가 존재합니다: {extracted_dir} (압축 해제 생략)")
    SATELLITE_IMG_DIR = extracted_dir
elif os.path.exists(zip_path):
    print(f"📦 {zip_path} 압축 파일 감지. 압축 해제를 시작합니다...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        for file in tqdm(file_list, desc="위성 이미지 압축 해제 중"):
            zip_ref.extract(member=file, path=extracted_dir)
    print(f"✅ 압축 해제 완료: {extracted_dir}")
    SATELLITE_IMG_DIR = extracted_dir
else:
    print("⚠️ jeju_dataset.zip 파일을 찾을 수 없습니다. 기본 satellite 폴더로 진행합니다.")
    SATELLITE_IMG_DIR = SATELLITE_DIR

# ── 이중 폴더 구조 방어 코드 추가 ──────────────────
if os.path.exists(SATELLITE_IMG_DIR):
    double_dir = os.path.join(SATELLITE_IMG_DIR, 'jeju_dataset')
    if os.path.exists(double_dir) and os.path.isdir(double_dir):
        SATELLITE_IMG_DIR = double_dir
        print(f"📂 이중 폴더 구조가 감지되어 경로를 재조정했습니다: {SATELLITE_IMG_DIR}")

# 실제 존재하는 위성 이미지 개수 확인
image_files = [f for f in os.listdir(SATELLITE_IMG_DIR) if f.endswith(('.jpg', '.png'))] if os.path.exists(SATELLITE_IMG_DIR) else []
print(f"현재 위성 이미지 폴더 내 실제 파일 개수: {len(image_files)}개")


## 3. Multi-Modal Dataset & DataLoader 구축

In [ ]:
FEATURE_COLS = ['wind_spd', 'wind_sin', 'wind_cos', 'temp', 'humidity', 'pressure', 'cloud', 'sunshine', 'solar_rad']
TARGET_COLS = ['solar_mw', 'wind_mw']
WINDOW_SIZE = 24

# ── 실제 위성 이미지가 존재하는 시점만 매칭하여 누락 데이터 필터링 ──
full_df['ts_str'] = pd.to_datetime(full_df['timestamp']).dt.strftime('%Y%m%d_%H%M')
existing_image_files = set([os.path.splitext(f)[0] for f in os.listdir(SATELLITE_IMG_DIR) if f.endswith(('.jpg', '.png'))])
full_df = full_df[full_df['ts_str'].isin(existing_image_files)].reset_index(drop=True)
full_df.drop(columns=['ts_str'], inplace=True)
print(f"필터링 후 남은 데이터 수: {len(full_df)}개 (실제 이미지가 매칭된 데이터만 사용)")

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()
full_df[FEATURE_COLS] = feature_scaler.fit_transform(full_df[FEATURE_COLS])
full_df[TARGET_COLS] = target_scaler.fit_transform(full_df[TARGET_COLS])

pickle.dump(feature_scaler, open(f'{BASE_DIR}/models/feature_scaler.pkl', 'wb'))
pickle.dump(target_scaler, open(f'{BASE_DIR}/models/target_scaler.pkl', 'wb'))

In [ ]:
class MultiModalEnergyDataset(Dataset):
    def __init__(self, df, feature_cols, target_cols, window_size=24, img_dir='', transform=None):
        self.df = df
        self.feature_cols = feature_cols
        self.target_cols = target_cols
        self.window_size = window_size
        self.img_dir = img_dir
        self.transform = transform
        
        self.features = df[feature_cols].values
        self.targets = df[target_cols].values
        self.timestamps = df['timestamp'].values
        
    def __len__(self):
        return len(self.df) - self.window_size
        
    def __getitem__(self, idx):
        # 1. 24시간 수치 시퀀스 데이터 추출 (t-23 ~ t)
        x_numeric = self.features[idx : idx + self.window_size]
        
        # 2. 타겟 시점(t) 위성 이미지 로딩
        target_ts = pd.to_datetime(self.timestamps[idx + self.window_size])
        ts_str = target_ts.strftime('%Y%m%d_%H%M')
        img_path = os.path.join(self.img_dir, f"{ts_str}.jpg")
        
        # 이미지가 없으면 임시 블랙 이미지 반환
        if os.path.exists(img_path):
            img = Image.open(img_path).convert('RGB')
        else:
            img = Image.new('RGB', (112, 112), color=0)
            
        if self.transform:
            x_image = self.transform(img)
        else:
            x_image = transforms.ToTensor()(img)
            
        y = self.targets[idx + self.window_size]
        return torch.tensor(x_numeric, dtype=torch.float32), x_image, torch.tensor(y, dtype=torch.float32)


In [ ]:
# ── 데이터 분할 및 DataLoader 빌드 (train_loader 정의) ───────────────────────
# 이미지 변형 파이프라인 (ResNet18 입력 조건인 112x112 크롭 및 Normalization)
img_transforms = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Train / Val / Test 순차적 분할
n = len(full_df) - WINDOW_SIZE
train_split = int(n * 0.70)
val_split   = int(n * 0.85)

train_df = full_df.iloc[:train_split + WINDOW_SIZE].reset_index(drop=True)
val_df   = full_df.iloc[train_split:val_split + WINDOW_SIZE].reset_index(drop=True)
test_df  = full_df.iloc[val_split:].reset_index(drop=True)

# MultiModal 커스텀 데이터셋 인스턴스 생성
train_dataset = MultiModalEnergyDataset(train_df, FEATURE_COLS, TARGET_COLS, WINDOW_SIZE, SATELLITE_IMG_DIR, img_transforms)
val_dataset   = MultiModalEnergyDataset(val_df, FEATURE_COLS, TARGET_COLS, WINDOW_SIZE, SATELLITE_IMG_DIR, img_transforms)
test_dataset  = MultiModalEnergyDataset(test_df, FEATURE_COLS, TARGET_COLS, WINDOW_SIZE, SATELLITE_IMG_DIR, img_transforms)

# DataLoader 선언
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ DataLoader 빌드 완료!")
print(f"  - Train samples: {len(train_dataset):,}")
print(f"  - Val samples:   {len(val_dataset):,}")
print(f"  - Test samples:  {len(test_dataset):,}")

## 4. Multi-Modal Fusion 모델 아키텍처 (ResNet18 + LSTM)

In [ ]:
class MultiModalFusionModel(nn.Module):
    def __init__(self, input_size=9, hidden_size=128, output_size=2):
        super(MultiModalFusionModel, self).__init__()
        
        # 1. 기상 시계열 LSTM Branch
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(hidden_size, 64, batch_first=True)
        self.dropout2 = nn.Dropout(0.2)
        
        # 2. 위성 이미지 CNN Branch (Pretrained ResNet18 백본)
        resnet = models.resnet18(pretrained=True)
        # 마지막 Fully Connected 레이어를 걷어내고 임베딩 출력으로 사용
        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # CNN 임베딩 벡터 차원 축소 (512 -> 64)
        self.cnn_fc = nn.Sequential(
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 3. Fusion Output Layer
        # LSTM 특징(64) + CNN 특징(64) = Concat 특징(128)
        self.fusion_fc = nn.Sequential(
            nn.Linear(64 + 64, 64),
            nn.ReLU(),
            nn.Linear(64, output_size)
        )
        
    def forward(self, x_numeric, x_image, sunshine_mask=None):
        # A. LSTM Branch
        out_num, _ = self.lstm1(x_numeric)
        out_num = self.dropout1(out_num)
        out_num, _ = self.lstm2(out_num)
        feat_numeric = self.dropout2(out_num[:, -1, :]) # (batch_size, 64)
        
        # B. CNN Branch
        feat_img = self.cnn_backbone(x_image) # (batch_size, 512, 1, 1)
        feat_img = torch.flatten(feat_img, 1) # (batch_size, 512)
        feat_img = self.cnn_fc(feat_img)       # (batch_size, 64)
        
        # C. Fusion
        fused = torch.cat([feat_numeric, feat_img], dim=1) # (batch_size, 128)
        pred = self.fusion_fc(fused) # (batch_size, 2)
        
        # D. 물리 마스킹
        if sunshine_mask is not None:
            pred[:, 0] = pred[:, 0] * sunshine_mask
            
        return pred

model = MultiModalFusionModel().to(DEVICE)
print(model)

## 5. 학습, 조기 종료 & 모델 평가

In [ ]:
class HybridLoss(nn.Module):
    def __init__(self, alpha=0.7):
        super().__init__()
        self.alpha = alpha
        self.mae = nn.L1Loss()
        self.mse = nn.MSELoss()
    def forward(self, pred, target):
        return self.alpha * self.mae(pred, target) + (1 - self.alpha) * self.mse(pred, target)

criterion = HybridLoss(alpha=0.7)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [ ]:
# ── tqdm 진행 표시줄이 포함된 학습 루프 ───────────────────────────
from tqdm.notebook import tqdm
import time

def get_sunshine_mask(X_batch):
    # FEATURE_COLS 기준 sunshine index = 7
    sunshine = X_batch[:, -1, 7]
    return (sunshine > 0.01).float()

EPOCHS = 50
PATIENCE = 7
best_val_loss = float('inf')
patience_cnt = 0
SAVE_PATH = f'{BASE_DIR}/models/best_model.pth'

train_losses, val_losses = [], []

# Epoch 단위 진행 표시줄
epoch_bar = tqdm(range(1, EPOCHS + 1), desc="전체 학습 진행률")

for epoch in epoch_bar:
    epoch_start_time = time.time()
    
    # ── Train ──
    model.train()
    train_loss = 0.0
    # 배치 단위 진행 표시줄 (내부 루프)
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch} 훈련 중", leave=False)
    for x_num, x_img, yb in train_bar:
        x_num, x_img, yb = x_num.to(DEVICE), x_img.to(DEVICE), yb.to(DEVICE)
        mask = get_sunshine_mask(x_num)
        
        optimizer.zero_grad()
        pred = model(x_num, x_img, sunshine_mask=mask)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(yb)
        train_bar.set_postfix(batch_loss=f"{loss.item():.4f}")
    train_loss /= len(train_dataset)
    
            
    # ── Validation ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_num, x_img, yb in val_loader:
            x_num, x_img, yb = x_num.to(DEVICE), x_img.to(DEVICE), yb.to(DEVICE)
            mask = get_sunshine_mask(x_num)
            pred = model(x_num, x_img, sunshine_mask=mask)
            val_loss += criterion(pred, yb).item() * len(yb)
    val_loss /= len(val_dataset)
    
    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    epoch_time = time.time() - epoch_start_time
    
    # 전체 진행 표시줄 텍스트 업데이트
    epoch_bar.set_postfix({
        'Train Loss': f"{train_loss:.4f}",
        'Val Loss': f"{val_loss:.4f}",
        'Sec/Epoch': f"{epoch_time:.1f}s"
    })
    
    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt = 0
        torch.save(model.state_dict(), SAVE_PATH)
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"
Early stopping triggered at Epoch {epoch}")
            break

In [ ]:
# 모델 평가
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

preds, trues = [], []
with torch.no_grad():
    for x_num, x_img, yb in test_loader:
        x_num, x_img = x_num.to(DEVICE), x_img.to(DEVICE)
        mask = get_sunshine_mask(x_num)
        pred = model(x_num, x_img, sunshine_mask=mask)
        preds.append(pred.cpu().numpy())
        trues.append(yb.numpy())

preds = np.concatenate(preds)
trues = np.concatenate(trues)

pred_orig = target_scaler.inverse_transform(preds)
true_orig = target_scaler.inverse_transform(trues)

print("=== 테스트 세트 평가 지표 ===")
print(f"태양광 MAE: {mean_absolute_error(true_orig[:, 0], pred_orig[:, 0]):.2f} MW (목표 <= 15)")
print(f"풍력 MAE: {mean_absolute_error(true_orig[:, 1], pred_orig[:, 1]):.2f} MW (목표 <= 15)")

## 6. RAG 벡터 DB (FAISS) 구축

In [ ]:
import faiss

# 훈련 데이터의 24시간 윈도우 수치(9*24=216) 추출
X_train_list = []
for i in range(len(train_dataset)):
    x_num, _, _ = train_dataset[i]
    X_train_list.append(x_num.numpy().flatten())
    
X_train_vecs = np.array(X_train_list, dtype=np.float32)
faiss.normalize_L2(X_train_vecs)

dim = X_train_vecs.shape[1] # 216
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(X_train_vecs)

# 메타데이터 매핑용 저장
train_timestamps = train_df['timestamp'].values[WINDOW_SIZE:]
train_solar = target_scaler.inverse_transform(train_dataset.targets[WINDOW_SIZE:])[:, 0]
train_wind = target_scaler.inverse_transform(train_dataset.targets[WINDOW_SIZE:])[:, 1]

faiss_meta = pd.DataFrame({
    'timestamp': train_timestamps,
    'solar_mw': train_solar,
    'wind_mw': train_wind
})

faiss.write_index(faiss_index, f'{BASE_DIR}/models/faiss_index.bin')
faiss_meta.to_pickle(f'{BASE_DIR}/models/faiss_meta.pkl')
print(f"FAISS DB 저장 완료: {faiss_index.ntotal}개 벡터")

## 7. 지식 그래프 (NetworkX KG) 구축

In [ ]:
import networkx as nx

def build_kg(df, f_scaler, t_scaler):
    G = nx.DiGraph()
    raw_feats = f_scaler.inverse_transform(df[FEATURE_COLS].values)
    raw_tgts = t_scaler.inverse_transform(df[TARGET_COLS].values)
    
    for i, row in df.iterrows():
        ts = str(row['timestamp'])
        snap_id = f"snap_{ts}"
        gen_id = f"gen_{ts}"
        
        G.add_node(snap_id, type='WeatherSnapshot', wind_spd=raw_feats[i, 0], cloud=raw_feats[i, 6], sunshine=raw_feats[i, 7])
        G.add_node(gen_id, type='GenerationRecord', solar_mw=raw_tgts[i, 0], wind_mw=raw_tgts[i, 1])
        G.add_edge(snap_id, gen_id, rel='RECORDED_AT')
        
        # 기상 이변 감지 노드 연계
        if raw_feats[i, 0] >= 18.0:
            ev_id = f"ev_highwind_{ts}"
            G.add_node(ev_id, type='WeatherEvent', event_type='high_wind', severity='Warning')
            G.add_edge(snap_id, ev_id, rel='TRIGGERS')
            
    return G

kg_data = train_df.copy()
G = build_kg(kg_data, feature_scaler, target_scaler)

with open(f'{BASE_DIR}/models/kg_graph.pkl', 'wb') as f:
    pickle.dump(G, f)
print(f"지식 그래프 저장 완료 (노드: {G.number_of_nodes()}개)")

## 8. 예측설명 에이전트 파이프라인

In [ ]:
class EnergyForecastAgent:
    def __init__(self, model, faiss_index, faiss_meta, kg, f_scaler, t_scaler):
        self.model = model
        self.index = faiss_index
        self.meta = faiss_meta
        self.kg = kg
        self.f_scaler = f_scaler
        self.t_scaler = t_scaler
        
    def predict(self, window_numeric, img_tensor, timestamp_str):
        self.model.eval()
        x_num = torch.tensor(window_numeric, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        x_img = img_tensor.unsqueeze(0).to(DEVICE)
        
        # 1. 모델 예측
        with torch.no_grad():
            pred = self.model(x_num, x_img).cpu().numpy()
        pred_orig = self.t_scaler.inverse_transform(pred)[0]
        
        solar_pred = max(0.0, float(pred_orig[0]))
        wind_pred = max(0.0, float(pred_orig[1]))
        
        # 2. 물리 제약 마스킹
        raw_window = self.f_scaler.inverse_transform(window_numeric)
        sunshine_now = raw_window[-1, 7]
        wind_now = raw_window[-1, 0]
        warnings = []
        
        if sunshine_now <= 0.01:
            solar_pred = 0.0
            warnings.append("야간 무일조 조건: 태양광 예측 0 MW 물리 강제 적용")
        if wind_now < 3.0 or wind_now > 25.0:
            wind_pred = 0.0
            warnings.append(f"풍속 임계치(Cut-in/out) 이탈 ({wind_now:.1f} m/s): 풍력 예측 0 MW 강제 적용")
            
        # 3. RAG / KG 탐색
        flat_vec = window_numeric.flatten().astype(np.float32).reshape(1, -1)
        faiss.normalize_L2(flat_vec)
        _, indices = self.index.search(flat_vec, 3)
        
        similar_cases = []
        for idx in indices[0]:
            similar_cases.append(self.meta.iloc[idx].to_dict())
            
        return {
            'solar_mw': solar_pred,
            'wind_mw': wind_pred,
            'warnings': warnings,
            'similar_cases': similar_cases
        }

agent = EnergyForecastAgent(model, faiss_index, faiss_meta, G, feature_scaler, target_scaler)
print("에이전트 파이프라인 설정 완료!")

In [ ]:
# 제주도만 따로 크롭해서 학습 후 파인튜닝(이전에는 전체적인 이미지 패턴을 학습해서 정확도 내려감)
import os
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm.notebook import tqdm

# ── 1. 설정 및 시드 고정 ──────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_DIR = '/content/drive/MyDrive/jeju_energy'
SAVE_PATH = f'{BASE_DIR}/models/best_model.pth'
NEW_SAVE_PATH = f'{BASE_DIR}/models/best_model_cropped.pth'
SATELLITE_IMG_DIR = f'{BASE_DIR}/data/satellite/jeju_dataset'

# 제주도 영역 크롭 박스 정의 (left, top, right, bottom)
CROP_BOX = (237, 301, 275, 323)

# ── 2. Dataset 클래스 내 Crop 전처리 반영 ──────────────────────────────────
class CroppedMultiModalDataset(Dataset):
    def __init__(self, df, feature_cols, target_cols, window_size=24, img_dir='', transform=None, crop_box=None):
        self.df = df
        self.feature_cols = feature_cols
        self.target_cols = target_cols
        self.window_size = window_size
        self.img_dir = img_dir
        self.transform = transform
        self.crop_box = crop_box
        
        self.features = df[feature_cols].values
        self.targets = df[target_cols].values
        self.timestamps = df['timestamp'].values
        
    def __len__(self):
        return len(self.df) - self.window_size
        
    def __getitem__(self, idx):
        x_numeric = self.features[idx : idx + self.window_size]
        
        target_ts = pd.to_datetime(self.timestamps[idx + self.window_size])
        ts_str = target_ts.strftime('%Y%m%d_%H%M')
        img_path = os.path.join(self.img_dir, f"{ts_str}.jpg")
        
        if os.path.exists(img_path):
            img = Image.open(img_path).convert('RGB')
            # ★ 제주도 좌표 영역 크롭 적용
            if self.crop_box:
                img = img.crop(self.crop_box)
        else:
            img = Image.new('RGB', (112, 112), color=0)
            
        if self.transform:
            x_image = self.transform(img)
        else:
            x_image = transforms.ToTensor()(img)
            
        y = self.targets[idx + self.window_size]
        return torch.tensor(x_numeric, dtype=torch.float32), x_image, torch.tensor(y, dtype=torch.float32)

# ── 3. 기존 베스트 모델 아키텍처 로드 및 가중치 고정 (Freezing) ───────────────
class MultiModalFusionModel(nn.Module):
    def __init__(self, input_size=9, hidden_size=128, output_size=2):
        super(MultiModalFusionModel, self).__init__()
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(hidden_size, 64, batch_first=True)
        self.dropout2 = nn.Dropout(0.2)
        
        resnet = models.resnet18(pretrained=False)
        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        self.cnn_fc = nn.Sequential(
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        self.fusion_fc = nn.Sequential(
            nn.Linear(64 + 64, 64),
            nn.ReLU(),
            nn.Linear(64, output_size)
        )
        
    def forward(self, x_numeric, x_image, sunshine_mask=None):
        out_num, _ = self.lstm1(x_numeric)
        out_num = self.dropout1(out_num)
        out_num, _ = self.lstm2(out_num)
        feat_numeric = self.dropout2(out_num[:, -1, :])
        
        feat_img = self.cnn_backbone(x_image)
        feat_img = torch.flatten(feat_img, 1)
        feat_img = self.cnn_fc(feat_img)
        
        fused = torch.cat([feat_numeric, feat_img], dim=1)
        pred = self.fusion_fc(fused)
        
        if sunshine_mask is not None:
            pred[:, 0] = pred[:, 0] * sunshine_mask
        return pred

# ── 4. 가중치 로드 및 CNN/FC 레이어만 학습 활성화 (Fine-Tuning 설정) ───────────
model = MultiModalFusionModel().to(DEVICE)
if os.path.exists(SAVE_PATH):
    model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
    print("기존 가동 중인 모델 가중치를 성공적으로 로드했습니다.")
else:
    raise FileNotFoundError(f"기존 가중치 파일 {SAVE_PATH}이 존재하지 않습니다.")

# ★ 핵심: LSTM 파트(수치 기상 데이터 분석) 가중치 동결
# 수치 데이터 분석 파트는 그대로 두고 위성 이미지의 크롭 인식률만 집중 학습시킵니다.
for name, param in model.named_parameters():
    if "lstm" in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

print("가중치 고정 완료 (LSTM 동결, CNN 및 Fusion FC 레이어만 파인튜닝)")

# ── 5. 데이터 준비 및 미세조정(Fine-Tuning) 훈련 시작 ──────────────────────
# (기존 Colab 로드맵의 Scaler 파일 및 df 데이터셋 로드 가정)
# full_df, FEATURE_COLS, TARGET_COLS, feature_scaler, target_scaler 가 존재해야 합니다.

img_transforms = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

n = len(full_df) - 24
train_split = int(n * 0.70)
val_split   = int(n * 0.85)

train_df = full_df.iloc[:train_split + 24].reset_index(drop=True)
val_df   = full_df.iloc[train_split:val_split + 24].reset_index(drop=True)

train_dataset = CroppedMultiModalDataset(train_df, FEATURE_COLS, TARGET_COLS, 24, SATELLITE_IMG_DIR, img_transforms, CROP_BOX)
val_dataset   = CroppedMultiModalDataset(val_df, FEATURE_COLS, TARGET_COLS, 24, SATELLITE_IMG_DIR, img_transforms, CROP_BOX)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 파인튜닝용 Loss 및 Optimizer (일부 레이어만 최적화하도록 필터링)
criterion = nn.L1Loss() # MAE 기준 집중 최적화
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# 5 Epoch 가속 튜닝 실행
EPOCHS = 5
print("미세조정(Fine-Tuning)을 시작합니다. (목표 Epochs: 5)")
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x_num, x_img, yb in train_loader:
        x_num, x_img, yb = x_num.to(DEVICE), x_img.to(DEVICE), yb.to(DEVICE)
        
        optimizer.zero_grad()
        pred = model(x_num, x_img)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(yb)
        
    train_loss /= len(train_dataset)
    
    # 검증 오차 평가
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_num, x_img, yb in val_loader:
            x_num, x_img, yb = x_num.to(DEVICE), x_img.to(DEVICE), yb.to(DEVICE)
            pred = model(x_num, x_img)
            val_loss += criterion(pred, yb).item() * len(yb)
    val_loss /= len(val_dataset)
    
    print(f"Epoch {epoch}/{EPOCHS} - Train Loss (MAE): {train_loss:.5f} | Val Loss (MAE): {val_loss:.5f}")

# 새 가중치 파일로 교체 저장
torch.save(model.state_dict(), SAVE_PATH)
print(f"🎉 파인튜닝 완료 및 최적 가중치 덮어쓰기 완료: {SAVE_PATH}")
